# TrueCarry club/hosel/ball YOLO fine-tune (Colab GPU)
1. Runtime → Change runtime type → **GPU** (T4 is plenty).
2. Run the first cell, upload `yolo_dataset.zip` (made by `tools/experimental/export_yolo_dataset.py`).
3. Run all. ~15–25 min on T4. Downloads `ClubDetectorV2.mlpackage` at the end.
4. Drop the mlpackage into `BallStrikeCamera/Analysis/` (replace or add beside `ClubDetector.mlpackage`).

In [ ]:
!pip -q install ultralytics coremltools
from google.colab import files
up = files.upload()   # upload yolo_dataset.zip
!unzip -q yolo_dataset.zip -d /content/
!ls /content/yolo_dataset

In [ ]:
from ultralytics import YOLO
# Small model: 360px source frames, 3 classes, few hundred images — yolov8s
# would overfit; n is the right size and exports light for on-device.
model = YOLO('yolov8n.pt')
model.train(data='/content/yolo_dataset/data.yaml', epochs=120, imgsz=640,
            batch=32, patience=25, degrees=5, translate=0.08, scale=0.25,
            fliplr=0.0,  # play direction matters — never mirror
            project='/content/runs', name='club_v2')

In [ ]:
# Validate: per-class detection quality on held-out frames
metrics = YOLO('/content/runs/club_v2/weights/best.pt').val(data='/content/yolo_dataset/data.yaml')
print(dict(zip(['head','hosel','ball'], metrics.box.maps)))

In [ ]:
# Export CoreML (same raw-output shape family the app's detector code parses)
YOLO('/content/runs/club_v2/weights/best.pt').export(format='coreml', imgsz=640, nms=False)
import shutil
from google.colab import files as gfiles
shutil.make_archive('/content/ClubDetectorV2.mlpackage', 'zip', '/content/runs/club_v2/weights/best.mlpackage')
gfiles.download('/content/ClubDetectorV2.mlpackage.zip')